<a href="https://colab.research.google.com/github/chinmayamohapatra-pixel/Bharghavtracker/blob/main/Video_QC_Tool_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Video QC Tool
> Paste a Google Drive folder link · set expected resolution · click **Run QC** · download report

**How to use:** `Runtime → Run All` → paste your Drive link → click ▶ Run QC

In [ ]:
# @title ⚙️ Setup (runs automatically — do not edit)
# @markdown This cell installs dependencies and authenticates with Google.
!pip install opencv-python-headless pydrive2 openpyxl ipywidgets --quiet
print('✅ Dependencies ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.3 MB/s eta 0:00:00
✅ Dependencies ready


In [ ]:
# @title 🔐 Authenticate with Google (runs automatically)
from google.colab import auth
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
print('✅ Authenticated with Google — ready to run QC')

In [ ]:
# @title 🎬 Video QC Tool
import cv2, numpy as np, os, re
from datetime import datetime
from IPython.display import display, clear_output, HTML, Javascript
import ipywidgets as widgets
import pandas as pd
from google.colab import files
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

DOWNLOAD_PATH = "/content/videos"
os.makedirs(DOWNLOAD_PATH, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def extract_folder_id(link):
    for pat in [r'/folders/([a-zA-Z0-9_-]+)', r'id=([a-zA-Z0-9_-]+)', r'^([a-zA-Z0-9_-]{25,})$']:
        m = re.search(pat, link.strip())
        if m: return m.group(1)
    return None

def format_duration(seconds):
    if seconds in ("N/A", "ERROR") or seconds is None: return "—"
    s = int(float(seconds))
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m:02d}:{sec:02d}"

def check_video(path, exp_w, exp_h):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): return "ERROR", "ERROR", "ERROR"
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur = round(tot / fps, 1) if fps > 0 else None
    cap.release()
    return (w == exp_w and h == exp_h), f"{w}×{h}", dur

# ── CSS injected once ─────────────────────────────────────────────────────────
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600&family=Inter:wght@400;500;600&display=swap');

.qc-root * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Inter', sans-serif; }

.qc-root {
  --bg: #0d1117; --surface: #161b22; --surface2: #21262d;
  --border: #30363d; --border2: #484f58;
  --text: #e6edf3; --muted: #7d8590;
  --purple: #6e40c9; --purple-l: #a78bfa;
  --green: #3fb950; --green-bg: #0f2a12; --green-bd: #1a4120;
  --red: #f85149;   --red-bg:   #2d1216; --red-bd:   #4a1e24;
  --blue: #58a6ff;  --blue-bg:  #0c1f3a; --blue-bd:  #1a3a6c;
  background: var(--bg); color: var(--text);
  border-radius: 16px; overflow: hidden;
  border: 1px solid var(--border); margin: 8px 0;
}

/* Header */
.qc-header {
  background: var(--surface);
  border-bottom: 1px solid var(--border);
  padding: 20px 28px; display: flex; align-items: center; gap: 14px;
}
.qc-logo {
  width: 42px; height: 42px; background: var(--purple);
  border-radius: 10px; display: flex; align-items: center;
  justify-content: center; font-size: 22px; flex-shrink: 0;
}
.qc-header-text h1 { font-size: 17px; font-weight: 600; color: var(--text); }
.qc-header-text p  { font-size: 12px; color: var(--muted); margin-top: 2px; }

/* Body */
.qc-body { padding: 24px 28px; }

/* URL input */
.qc-input-label {
  font-size: 11px; font-weight: 600; color: var(--muted);
  text-transform: uppercase; letter-spacing: .8px; margin-bottom: 8px;
}
.qc-url-row { display: flex; gap: 10px; align-items: stretch; margin-bottom: 20px; }
.qc-url-input {
  flex: 1; background: var(--surface); border: 1px solid var(--border2);
  color: var(--text); border-radius: 8px; padding: 10px 14px;
  font-size: 14px; outline: none; font-family: 'IBM Plex Mono', monospace;
  transition: border-color .15s;
}
.qc-url-input:focus { border-color: var(--purple); }
.qc-url-input::placeholder { color: var(--muted); }
.qc-run-btn {
  background: var(--purple); color: #fff; border: none;
  border-radius: 8px; padding: 0 22px; font-size: 13px; font-weight: 600;
  cursor: pointer; white-space: nowrap; transition: opacity .15s;
}
.qc-run-btn:hover { opacity: .85; }
.qc-run-btn:disabled { opacity: .4; cursor: not-allowed; }

/* Settings row */
.qc-settings {
  display: flex; gap: 24px; align-items: center;
  background: var(--surface); border: 1px solid var(--border);
  border-radius: 8px; padding: 12px 16px; margin-bottom: 20px;
  flex-wrap: wrap;
}
.qc-settings-label { font-size: 12px; color: var(--muted); margin-right: 8px; }
.qc-settings input[type=number] {
  background: var(--bg); border: 1px solid var(--border2);
  color: var(--text); font-family: 'IBM Plex Mono', monospace;
  font-size: 13px; border-radius: 6px; padding: 5px 10px; width: 80px;
  outline: none;
}
.qc-settings input[type=number]:focus { border-color: var(--purple); }
.qc-sep { color: var(--muted); font-size: 18px; font-family: monospace; }
.qc-safe { margin-left: auto; font-size: 11px; color: var(--muted); }

/* Progress */
.qc-progress { margin-bottom: 20px; display: none; }
.qc-progress-bar-bg { background: var(--surface2); border-radius: 4px; height: 4px; overflow: hidden; margin-bottom: 8px; }
.qc-progress-bar-fill { height: 100%; background: var(--purple); border-radius: 4px; transition: width .3s; }
.qc-progress-label { font-size: 12px; color: var(--muted); font-family: 'IBM Plex Mono', monospace; }

/* Summary cards */
.qc-summary { display: none; grid-template-columns: repeat(4,1fr); gap: 10px; margin-bottom: 20px; }
.qc-stat { border-radius: 8px; padding: 14px 16px; text-align: center; border: 1px solid var(--border); background: var(--surface); }
.qc-stat.pass { background: var(--green-bg); border-color: var(--green-bd); }
.qc-stat.fail { background: var(--red-bg);   border-color: var(--red-bd); }
.qc-stat.dur  { background: var(--blue-bg);  border-color: var(--blue-bd); }
.qc-stat-num { font-family: 'IBM Plex Mono', monospace; font-size: 24px; font-weight: 600; }
.qc-stat.pass .qc-stat-num { color: var(--green); }
.qc-stat.fail .qc-stat-num { color: var(--red); }
.qc-stat.total .qc-stat-num { color: var(--purple-l); }
.qc-stat.dur  .qc-stat-num { color: var(--blue); font-size: 17px; }
.qc-stat-label { font-size: 10px; color: var(--muted); margin-top: 4px; text-transform: uppercase; letter-spacing: .6px; }

/* Results table */
.qc-results { display: none; }
.qc-results-hdr { display: flex; align-items: center; justify-content: space-between; margin-bottom: 10px; }
.qc-results-title { font-size: 11px; font-weight: 600; color: var(--muted); text-transform: uppercase; letter-spacing: .8px; }
.qc-dl-group { display: flex; gap: 8px; }
.qc-dl-btn {
  background: var(--surface); border: 1px solid var(--border2);
  color: var(--text); border-radius: 6px; padding: 5px 12px;
  font-size: 11px; font-weight: 500; cursor: pointer;
  transition: border-color .15s, background .15s;
}
.qc-dl-btn:hover { border-color: var(--purple); background: var(--surface2); }

.qc-table-wrap { border: 1px solid var(--border); border-radius: 10px; overflow: hidden; }
.qc-table { width: 100%; border-collapse: collapse; font-family: 'IBM Plex Mono', monospace; }
.qc-table thead th {
  padding: 10px 14px; text-align: left; font-size: 10px; font-weight: 600;
  color: var(--muted); text-transform: uppercase; letter-spacing: .8px;
  background: var(--surface2); border-bottom: 1px solid var(--border);
}
.qc-table tbody tr { border-bottom: 1px solid var(--border); }
.qc-table tbody tr:last-child { border-bottom: none; }
.qc-table tbody tr:hover { background: var(--surface2); }
.qc-table tbody td { padding: 11px 14px; font-size: 12px; vertical-align: middle; color: var(--text); }
.qc-fname { font-family: 'Inter', sans-serif; font-size: 13px; max-width: 260px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.qc-badge { display: inline-flex; align-items: center; gap: 4px; font-size: 10px; font-weight: 600; padding: 3px 9px; border-radius: 5px; letter-spacing: .3px; }
.qc-badge.pass { background: var(--green-bg); color: var(--green); border: 1px solid var(--green-bd); }
.qc-badge.fail { background: var(--red-bg);   color: var(--red);   border: 1px solid var(--red-bd); }
.qc-res-ok  { color: var(--green); }
.qc-res-bad { color: var(--red); }
.qc-dur     { color: var(--blue); }
.qc-muted   { color: var(--muted); }

/* Status bar */
.qc-status { font-size: 12px; color: var(--muted); font-family: 'IBM Plex Mono', monospace; min-height: 18px; }
.qc-status.ok  { color: var(--green); }
.qc-status.err { color: var(--red); }
.qc-status.warn { color: #e3b341; }

/* Reset btn */
.qc-reset-btn {
  margin-top: 16px; background: transparent; border: 1px solid var(--border);
  color: var(--muted); border-radius: 6px; padding: 6px 14px;
  font-size: 12px; cursor: pointer; display: none;
  transition: border-color .15s, color .15s;
}
.qc-reset-btn:hover { border-color: var(--border2); color: var(--text); }
</style>
"""))

# ── Build widget tree ─────────────────────────────────────────────────────────
W = widgets.HTML

# Header
header = W("""
<div class="qc-root">
  <div class="qc-header">
    <div class="qc-logo">🎬</div>
    <div class="qc-header-text">
      <h1>Video QC Tool</h1>
      <p>Paste a Google Drive folder link · set resolution · run</p>
    </div>
  </div>
</div>""")

# URL input (native HTML rendered inside Output widget via display)
url_input = widgets.Text(
    placeholder="https://drive.google.com/drive/folders/...",
    layout=widgets.Layout(width="100%"))

width_input  = widgets.BoundedIntText(value=1080, min=1, max=9999,
                layout=widgets.Layout(width="80px"))
height_input = widgets.BoundedIntText(value=1920, min=1, max=9999,
                layout=widgets.Layout(width="80px"))

run_btn   = widgets.Button(description="▶  Run QC",
              button_style="", layout=widgets.Layout(width="120px", height="40px"))
run_btn.style.button_color  = "#6e40c9"
run_btn.style.text_color    = "#ffffff"
run_btn.style.font_weight   = "600"

dl_csv  = widgets.Button(description="⬇ CSV",
            layout=widgets.Layout(width="90px", height="32px", display="none"))
dl_xlsx = widgets.Button(description="⬇ Excel",
            layout=widgets.Layout(width="90px", height="32px", display="none"))
reset_btn = widgets.Button(description="↺ Start over",
              layout=widgets.Layout(width="110px", height="32px", display="none"))
for b in [dl_csv, dl_xlsx, reset_btn]:
    b.style.button_color = "#21262d"
    b.style.text_color   = "#e6edf3"

progress_bar = widgets.IntProgress(value=0, min=0, max=100,
    layout=widgets.Layout(width="100%", height="4px", display="none"))
status_out  = widgets.Output()
summary_out = widgets.Output()
table_out   = widgets.Output()

# Inject custom style once; widgets handle the rest
display(HTML("""<style>
.widget-text input { background:#161b22!important; border:1px solid #484f58!important;
  color:#e6edf3!important; border-radius:8px!important; font-family:'IBM Plex Mono',monospace!important;
  font-size:13px!important; padding:8px 12px!important; }
.widget-button { border-radius:8px!important; font-family:'Inter',sans-serif!important; }
.widget-progress .progress-bar { background:#6e40c9!important; }
.widget-progress { border-radius:4px!important; background:#21262d!important; }
</style>"""))

ui = widgets.VBox([
    W("""<div style="background:#161b22;border:1px solid #30363d;border-radius:12px 12px 0 0;
         padding:18px 20px 0;margin-top:8px">
         <div style="font-size:10px;color:#7d8590;font-weight:600;text-transform:uppercase;letter-spacing:.8px;margin-bottom:8px">
         Drive Folder URL</div></div>"""),
    widgets.HBox([url_input, run_btn],
        layout=widgets.Layout(padding="0 20px", background_color="#161b22")),
    W("""<div style="background:#161b22;padding:0 20px 6px"></div>"""),
    widgets.HBox([
        W("<span style='font-size:11px;color:#7d8590;line-height:32px'>Expected resolution</span>"),
        width_input,
        W("<span style='font-size:16px;color:#7d8590;line-height:32px;padding:0 6px'>×</span>"),
        height_input,
        W("<span style='font-size:11px;color:#7d8590;line-height:32px;margin-left:4px'>(W × H)</span>"),
    ], layout=widgets.Layout(padding="10px 20px 14px", background_color="#161b22",
                              border_bottom="1px solid #30363d", gap="8px", align_items="center")),
    widgets.VBox([progress_bar, status_out],
        layout=widgets.Layout(padding="12px 20px 0", background_color="#0d1117")),
    summary_out,
    table_out,
    widgets.HBox([dl_csv, dl_xlsx, reset_btn],
        layout=widgets.Layout(padding="0 20px 20px", background_color="#0d1117", gap="8px")),
], layout=widgets.Layout(border="1px solid #30363d", border_radius="12px",
                          max_width="920px", margin="0 auto",
                          background_color="#0d1117"))

display(ui)

# ── State ─────────────────────────────────────────────────────────────────────
_cache = []

def set_status(msg, kind="muted"):
    color = {"ok":"#3fb950","err":"#f85149","warn":"#e3b341","muted":"#7d8590"}.get(kind,"#7d8590")
    with status_out:
        clear_output(wait=True)
        display(HTML(f"<span style='font-family:IBM Plex Mono,monospace;font-size:12px;color:{color}'>{msg}</span>"))

def render_summary(results):
    total  = len(results)
    passed = sum(1 for r in results if r["Status"]=="PASS")
    total_secs = sum(r["_dur_secs"] for r in results if r["_dur_secs"] is not None)
    total_dur  = format_duration(total_secs) if total_secs else "—"
    with summary_out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;padding:16px 20px 0;background:#0d1117">
          <div style="background:#0f2a12;border:1px solid #1a4120;border-radius:8px;padding:14px;text-align:center">
            <div style="font-family:'IBM Plex Mono',monospace;font-size:24px;font-weight:600;color:#3fb950">{passed}</div>
            <div style="font-size:10px;color:#7d8590;margin-top:4px;text-transform:uppercase;letter-spacing:.6px">✓ Passed</div>
          </div>
          <div style="background:#2d1216;border:1px solid #4a1e24;border-radius:8px;padding:14px;text-align:center">
            <div style="font-family:'IBM Plex Mono',monospace;font-size:24px;font-weight:600;color:#f85149">{total-passed}</div>
            <div style="font-size:10px;color:#7d8590;margin-top:4px;text-transform:uppercase;letter-spacing:.6px">✕ Failed</div>
          </div>
          <div style="background:#161b22;border:1px solid #30363d;border-radius:8px;padding:14px;text-align:center">
            <div style="font-family:'IBM Plex Mono',monospace;font-size:24px;font-weight:600;color:#a78bfa">{total}</div>
            <div style="font-size:10px;color:#7d8590;margin-top:4px;text-transform:uppercase;letter-spacing:.6px">Total episodes</div>
          </div>
          <div style="background:#0c1f3a;border:1px solid #1a3a6c;border-radius:8px;padding:14px;text-align:center">
            <div style="font-family:'IBM Plex Mono',monospace;font-size:17px;font-weight:600;color:#58a6ff">{total_dur}</div>
            <div style="font-size:10px;color:#7d8590;margin-top:4px;text-transform:uppercase;letter-spacing:.6px">⏱ Total duration</div>
          </div>
        </div>"""))

def render_table(results):
    rows = ""
    for i, r in enumerate(results):
        is_pass = r["Status"] == "PASS"
        badge   = f'<span style="display:inline-flex;align-items:center;font-size:10px;font-weight:600;padding:3px 9px;border-radius:5px;font-family:\'IBM Plex Mono\',monospace;background:{"#0f2a12" if is_pass else "#2d1216"};color:{"#3fb950" if is_pass else "#f85149"};border:1px solid {"#1a4120" if is_pass else "#4a1e24"}">{"✓ PASS" if is_pass else "✕ FAIL"}</span>'
        res_col = f'<span style="color:{"#3fb950" if is_pass else "#f85149"}">{r["Actual Resolution"]}</span>'
        bg = "#0d1117" if i % 2 == 0 else "#0f1318"
        rows += f"""<tr style="border-bottom:1px solid #30363d;background:{bg}">
          <td style="padding:11px 14px;font-family:Inter,sans-serif;font-size:13px;max-width:260px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;color:#e6edf3" title="{r['File Name']}">{r['File Name']}</td>
          <td style="padding:11px 14px">{badge}</td>
          <td style="padding:11px 14px;font-family:'IBM Plex Mono',monospace;font-size:12px">{res_col}</td>
          <td style="padding:11px 14px;font-family:'IBM Plex Mono',monospace;font-size:12px;color:#58a6ff">{r['Duration (HH:MM:SS)']}</td>
          <td style="padding:11px 14px;font-family:'IBM Plex Mono',monospace;font-size:11px;color:#7d8590">{r['Checked At']}</td>
        </tr>"""
    with table_out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="padding:16px 20px 0;background:#0d1117">
          <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:10px">
            <span style="font-size:10px;font-weight:600;color:#7d8590;text-transform:uppercase;letter-spacing:.8px">Results</span>
          </div>
          <div style="border:1px solid #30363d;border-radius:10px;overflow:hidden">
            <table style="width:100%;border-collapse:collapse">
              <thead><tr style="background:#21262d">
                <th style="padding:10px 14px;text-align:left;font-size:10px;font-weight:600;color:#7d8590;text-transform:uppercase;letter-spacing:.8px;border-bottom:1px solid #30363d">File name</th>
                <th style="padding:10px 14px;text-align:left;font-size:10px;font-weight:600;color:#7d8590;text-transform:uppercase;letter-spacing:.8px;border-bottom:1px solid #30363d">Status</th>
                <th style="padding:10px 14px;text-align:left;font-size:10px;font-weight:600;color:#7d8590;text-transform:uppercase;letter-spacing:.8px;border-bottom:1px solid #30363d">Resolution</th>
                <th style="padding:10px 14px;text-align:left;font-size:10px;font-weight:600;color:#7d8590;text-transform:uppercase;letter-spacing:.8px;border-bottom:1px solid #30363d">Duration</th>
                <th style="padding:10px 14px;text-align:left;font-size:10px;font-weight:600;color:#7d8590;text-transform:uppercase;letter-spacing:.8px;border-bottom:1px solid #30363d">Checked at</th>
              </tr></thead>
              <tbody>{rows}</tbody>
            </table>
          </div>
        </div>"""))

# ── Run handler ───────────────────────────────────────────────────────────────
def on_run(b):
    global _cache
    link = url_input.value.strip()
    if not link:
        set_status("⚠ Paste a Drive folder link first.", "warn"); return
    fid = extract_folder_id(link)
    if not fid:
        set_status("✕ Invalid Drive link — use a folder URL.", "err"); return

    run_btn.disabled = True
    for btn in [dl_csv, dl_xlsx, reset_btn]: btn.layout.display = "none"
    with summary_out: clear_output()
    with table_out:   clear_output()
    _cache = []

    try:
        set_status("🔍 Scanning Drive folder…")
        progress_bar.layout.display = ""
        progress_bar.value = 0

        file_list = drive.ListFile({'q': f"'{fid}' in parents and trashed=false"}).GetList()
        videos = [f for f in file_list if f['title'].lower().endswith((".mp4",".mov",".mkv"))]

        if not videos:
            set_status("⚠ No .mp4 / .mov / .mkv files found in that folder.", "warn")
            progress_bar.layout.display = "none"; run_btn.disabled = False; return

        total = len(videos)
        exp_w = width_input.value; exp_h = height_input.value
        results = []

        for i, f in enumerate(videos):
            fname = f['title']; fpath = os.path.join(DOWNLOAD_PATH, fname)
            progress_bar.value = int(i / total * 90)
            set_status(f"⬇ Downloading {i+1}/{total}: {fname}")
            f.GetContentFile(fpath)
            set_status(f"🔍 Checking {i+1}/{total}: {fname}")
            res_ok, actual_res, dur_secs = check_video(fpath, exp_w, exp_h)
            status = "PASS" if res_ok is True else "FAIL"
            results.append({
                "File Name":          fname,
                "Status":             status,
                "Resolution OK":      "Yes" if res_ok is True else ("ERROR" if res_ok=="ERROR" else "No"),
                "Actual Resolution":  actual_res,
                "Duration (s)":       dur_secs if dur_secs is not None else "N/A",
                "Duration (HH:MM:SS)":format_duration(dur_secs),
                "Checked At":         datetime.now().strftime("%Y-%m-%d %H:%M"),
                "_dur_secs":          dur_secs,
            })
            os.remove(fpath)

        progress_bar.value = 100
        _cache = results
        passed = sum(1 for r in results if r["Status"]=="PASS")
        set_status(f"✓ Done — {passed}/{total} passed · {total-passed} failed",
                   "ok" if passed == total else "warn")

        render_summary(results)
        render_table(results)
        for btn in [dl_csv, dl_xlsx, reset_btn]: btn.layout.display = ""

    except Exception as e:
        set_status(f"✕ Error: {e}", "err")
        progress_bar.layout.display = "none"
    finally:
        run_btn.disabled = False

# ── Export handlers ───────────────────────────────────────────────────────────
def on_csv(b):
    if not _cache: return
    export = [{k: v for k, v in r.items() if k != "_dur_secs"} for r in _cache]
    fname = f"video_qc_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    pd.DataFrame(export).to_csv(fname, index=False)
    files.download(fname)

def on_xlsx(b):
    if not _cache: return
    export = [{k: v for k, v in r.items() if k != "_dur_secs"} for r in _cache]
    wb = Workbook(); ws = wb.active; ws.title = "QC Report"
    hf = PatternFill("solid", fgColor="1a237e")
    pf = PatternFill("solid", fgColor="0f2a12")
    ff = PatternFill("solid", fgColor="2d1216")
    af = PatternFill("solid", fgColor="0d1117")
    th = Side(style="thin", color="30363d")
    bd = Border(left=th, right=th, top=th, bottom=th)
    hdrs = list(export[0].keys())
    for c, h in enumerate(hdrs, 1):
        cell = ws.cell(row=1, column=c, value=h)
        cell.fill = hf; cell.font = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = bd
    ws.row_dimensions[1].height = 28
    for ri, rec in enumerate(export, 2):
        ip = rec["Status"] == "PASS"
        for ci, key in enumerate(hdrs, 1):
            cell = ws.cell(row=ri, column=ci, value=rec[key]); cell.border = bd
            cell.alignment = Alignment(horizontal="left" if ci==1 else "center", vertical="center")
            if key == "Status":
                cell.fill = pf if ip else ff
                cell.font = Font(bold=True, color="3fb950" if ip else "f85149", size=11)
            elif ri % 2 == 0:
                cell.fill = af
        ws.row_dimensions[ri].height = 22
    for i, w in enumerate([38, 8, 14, 12, 16, 18, 18], 1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.freeze_panes = "A2"
    fname = f"video_qc_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
    wb.save(fname); files.download(fname)

def on_reset(b):
    global _cache; _cache = []
    url_input.value = ""
    with summary_out: clear_output()
    with table_out:   clear_output()
    with status_out:  clear_output()
    progress_bar.value = 0; progress_bar.layout.display = "none"
    for btn in [dl_csv, dl_xlsx, reset_btn]: btn.layout.display = "none"

run_btn.on_click(on_run)
dl_csv.on_click(on_csv)
dl_xlsx.on_click(on_xlsx)
reset_btn.on_click(on_reset)
